# 🚢 Titanic Baseline Notebook

**Competition:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)  
**Goal:** Predict which passengers survived the Titanic shipwreck.  
**Metric:** Binary classification accuracy on the test set.

---

> **Kernel:** Make sure you have selected the `Python (titanic-ml)` kernel before running any cells.
> In VS Code, click the kernel selector in the top-right corner of this notebook.

## 1. 📦 Imports

- We import libraries that help us read data, inspect data, and later build ML models.
- **pandas** is used for CSV and dataframe work.
- **numpy** is used for numerical operations.
- **scikit-learn** (imported as `sklearn`) is imported only to confirm the setup for later.

In [3]:
import pandas as pd
import numpy as np
import sklearn

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Notebook is connected correctly.")

pandas: 3.0.3
numpy: 2.4.6
scikit-learn: 1.8.0
Notebook is connected correctly.


## 2. 📂 Load Data

- Loading data means reading CSV files from the data folder into Python variables.
- The CSV files remain in `data/`.
- The notebook reads them into memory so Python can work with them.

### Expected learning point:
- `train.csv` should have 891 rows.
- `test.csv` should have 418 rows.
- `train.csv` contains the target column `Survived`.
- `test.csv` does **not** contain the `Survived` column.

In [4]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
gender_submission = pd.read_csv("data/gender_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", gender_submission.shape)

train.head()

Train shape: (891, 12)
Test shape: (418, 11)
Sample submission shape: (418, 2)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. 🧠 Problem Understanding

This is a **supervised machine learning classification** problem.

The model will learn from `train.csv` because `train.csv` contains the correct answer column: `Survived`.

The model will later predict `Survived` for passengers in `test.csv`.

**Target:**
- `Survived`

**Target values:**
- `0` = did not survive
- `1` = survived

This is **binary classification** because there are only two possible outcomes.

## 4. 🔍 Data Inspection

- Before cleaning or modeling, we inspect the dataset.
- We need to understand columns, data types, and sample values.

In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [12]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 5. 🧩 Missing Values

- Missing values are empty cells.
- Many ML models cannot work directly with missing values.
- Before training, we need to know which columns have missing data.

In [8]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [9]:
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         1
dtype: int64

## 6. 🎯 Target Distribution

- The target column is `Survived`.
- We check how many passengers survived and how many did not.
- This helps us understand whether the dataset is balanced or unbalanced.

In [13]:
train["Survived"].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [14]:
train["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## 7. 📏 Features vs Target

- **Features** are the input columns used to make predictions.
- **Target** is the answer the model is trying to predict.
- In this project, the target is `Survived`.

In [11]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = train[features]
y = train["Survived"]

print("Feature data shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature data shape: (891, 7)
Target shape: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


- `X` contains the input features.
- `y` contains the correct answers.
- Later, the model will learn patterns from `X` to predict `y`.

## 8. 🚺 Simple Gender Baseline

- Before machine learning, we create a simple baseline.
- A baseline is a simple starting solution.
- In the Titanic competition, a common simple rule is:
  - If the passenger is female, predict survived (`1`).
  - If the passenger is male, predict did not survive (`0`).
- This is not our final model. It is a comparison point.

In [15]:
gender_baseline = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": (test["Sex"] == "female").astype(int)
})

gender_baseline.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [16]:
gender_baseline.shape

(418, 2)

## 9. 💾 Save Gender Baseline Submission

- Kaggle submission files must contain only `PassengerId` and `Survived`.
- No extra columns should be included.
- This creates our first simple Kaggle-ready submission file.

In [19]:
gender_baseline.to_csv("submissions/gender_baseline_submission.csv", index=False)

print("Gender baseline submission created successfully.")

Gender baseline submission created successfully.


In [18]:
check_submission = pd.read_csv("submissions/gender_baseline_submission.csv")

print(check_submission.shape)
check_submission.head()

(418, 2)


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## 10. 📝 Learning Checkpoint Summary

Before moving to model training, I should be able to answer:

1. What is the difference between `train.csv` and `test.csv`?
2. What does the `Survived` column mean?
3. Why is this a classification problem?
4. What are features?
5. What is the target?
6. What does it mean to load data into a notebook?
7. Why do we inspect missing values before training?
8. What is a baseline?
9. Why is the gender baseline useful?
10. What format does Kaggle expect for submission?